# Small DDPM from Scratch — CIFAR-10

A minimal, unconditional Denoising Diffusion Probabilistic Model (Ho et al. 2020), trained from scratch on CIFAR-10 at 32x32.

**What this teaches you (the actual mechanics, not a wrapper around a pretrained model):**
- The forward diffusion process (closed-form noising at any timestep `t`)
- A U-Net that predicts the noise added at timestep `t`
- The training objective (simple noise-prediction MSE — no adversarial loss)
- Reverse sampling: iteratively denoising from pure noise back to an image

Runs on a single T4. Includes checkpointing to Google Drive since Colab sessions time out.


## 0. Setup

In [ ]:
!pip install -q torchvision tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import math
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
# Optional: mount Google Drive for checkpointing across Colab session timeouts
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = "/content/drive/MyDrive/ddpm_cifar10_ckpts"
os.makedirs(CKPT_DIR, exist_ok=True)


## 1. Config

Base channels 64, ~40M params — comparable scale to a small GPT, trains in a few hours on T4.

In [ ]:
class Config:
    image_size = 32
    channels = 3
    batch_size = 128
    base_channels = 64
    channel_mults = (1, 2, 2, 2)     # resolution levels: 32 -> 16 -> 8 -> 4
    attn_resolutions = (8,)          # self-attention at 8x8
    num_res_blocks = 2
    timesteps = 1000                 # T in the DDPM paper
    beta_start = 1e-4
    beta_end = 0.02
    lr = 2e-4
    ema_decay = 0.9999
    epochs = 200                     # adjust based on time budget; checkpoint every epoch
    grad_accum_steps = 1
    mixed_precision = True

cfg = Config()


## 2. Forward diffusion process

The core trick of DDPM: instead of adding noise step by step 1000 times, we can jump directly
to any timestep `t` in closed form:

`x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise`

where `alpha_bar_t` is the cumulative product of `(1 - beta_t)` up to step `t`.
This closed form is what makes training tractable — every training step samples a random `t`
and jumps straight there, rather than simulating the whole chain.

In [ ]:
def linear_beta_schedule(timesteps, beta_start, beta_end):
    return torch.linspace(beta_start, beta_end, timesteps)

betas = linear_beta_schedule(cfg.timesteps, cfg.beta_start, cfg.beta_end).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

# Used in the reverse process (posterior variance)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

def extract(a, t, x_shape):
    # Pull out the right timestep's scalar for each item in the batch, reshape to broadcast
    batch_size = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))

def q_sample(x0, t, noise=None):
    """Forward process: noise x0 to timestep t in one closed-form step."""
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_ac = extract(sqrt_alphas_cumprod, t, x0.shape)
    sqrt_omac = extract(sqrt_one_minus_alphas_cumprod, t, x0.shape)
    return sqrt_ac * x0 + sqrt_omac * noise


### Sanity check: visualize an image getting progressively noised
If this looks right (clean -> pure static), the forward process is implemented correctly.

In [ ]:
transform = T.Compose([T.ToTensor(), T.Lambda(lambda x: x * 2 - 1)])  # scale to [-1, 1]
dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)

sample_img, _ = dataset[0]
sample_img = sample_img.unsqueeze(0).to(device)

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i, t_val in enumerate([0, 50, 150, 300, 600, 999]):
    t = torch.tensor([t_val], device=device)
    noised = q_sample(sample_img, t)
    img = (noised.squeeze(0).permute(1, 2, 0).cpu().clamp(-1, 1) + 1) / 2
    axes[i].imshow(img)
    axes[i].set_title(f"t={t_val}")
    axes[i].axis("off")
plt.suptitle("Forward diffusion process")
plt.show()


## 3. U-Net (the noise predictor)

This is the model being trained. At every step it takes a noisy image `x_t` and the timestep `t`,
and predicts the noise that was added. The timestep is injected into every residual block via a
sinusoidal embedding (same idea as positional encoding in transformers) so the network knows
"how noisy" the input currently is.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.block1 = nn.Sequential(nn.GroupNorm(8, in_ch), nn.SiLU(), nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.block2 = nn.Sequential(nn.GroupNorm(8, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.block1(x)
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.block2(h)
        return h + self.res_conv(x)


class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv(self.norm(x)).reshape(b, 3, c, h * w).permute(1, 0, 3, 2)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(c), dim=-1)
        out = (attn @ v).permute(0, 2, 1).reshape(b, c, h, w)
        return x + self.proj(out)


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x):
        return self.op(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)
    def forward(self, x):
        return self.op(x)


class UNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        ch = cfg.base_channels
        time_emb_dim = ch * 4
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(ch),
            nn.Linear(ch, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        self.init_conv = nn.Conv2d(cfg.channels, ch, 3, padding=1)

        # Downsampling path
        self.down_blocks = nn.ModuleList()
        cur_res = cfg.image_size
        in_ch = ch
        channels_at_each_level = [ch]
        for i, mult in enumerate(cfg.channel_mults):
            out_ch = ch * mult
            for _ in range(cfg.num_res_blocks):
                layers = [ResBlock(in_ch, out_ch, time_emb_dim)]
                if cur_res in cfg.attn_resolutions:
                    layers.append(SelfAttention(out_ch))
                self.down_blocks.append(nn.ModuleList(layers))
                in_ch = out_ch
                channels_at_each_level.append(in_ch)
            if i != len(cfg.channel_mults) - 1:
                self.down_blocks.append(nn.ModuleList([Downsample(in_ch)]))
                channels_at_each_level.append(in_ch)
                cur_res //= 2

        # Bottleneck
        self.mid_block1 = ResBlock(in_ch, in_ch, time_emb_dim)
        self.mid_attn = SelfAttention(in_ch)
        self.mid_block2 = ResBlock(in_ch, in_ch, time_emb_dim)

        # Upsampling path
        self.up_blocks = nn.ModuleList()
        for i, mult in reversed(list(enumerate(cfg.channel_mults))):
            out_ch = ch * mult
            for _ in range(cfg.num_res_blocks + 1):
                skip_ch = channels_at_each_level.pop()
                layers = [ResBlock(in_ch + skip_ch, out_ch, time_emb_dim)]
                if cur_res in cfg.attn_resolutions:
                    layers.append(SelfAttention(out_ch))
                self.up_blocks.append(nn.ModuleList(layers))
                in_ch = out_ch
            if i != 0:
                self.up_blocks.append(nn.ModuleList([Upsample(in_ch)]))
                cur_res *= 2

        self.out_norm = nn.GroupNorm(8, in_ch)
        self.out_conv = nn.Conv2d(in_ch, cfg.channels, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        h = self.init_conv(x)
        skips = [h]

        for layers in self.down_blocks:
            if isinstance(layers[0], Downsample):
                h = layers[0](h)
            else:
                h = layers[0](h, t_emb)
                if len(layers) > 1:
                    h = layers[1](h)
            skips.append(h)

        h = self.mid_block1(h, t_emb)
        h = self.mid_attn(h)
        h = self.mid_block2(h, t_emb)

        for layers in self.up_blocks:
            if isinstance(layers[0], Upsample):
                h = layers[0](h)
            else:
                skip = skips.pop()
                h = torch.cat([h, skip], dim=1)
                h = layers[0](h, t_emb)
                if len(layers) > 1:
                    h = layers[1](h)

        h = self.out_norm(h)
        h = F.silu(h)
        return self.out_conv(h)


model = UNet(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.1f}M")


## 4. EMA (exponential moving average) of weights

Standard trick for diffusion models — sampling from the EMA weights gives noticeably better,
less noisy generations than the raw training weights.

In [ ]:
class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v, alpha=1 - self.decay)
            else:
                self.shadow[k] = v.clone()

    def apply_to(self, model):
        model.load_state_dict(self.shadow)

ema = EMA(model, cfg.ema_decay)


## 5. Training loop

Every step: sample a random timestep `t` per image, noise the image to that step, ask the model to predict the noise, MSE loss. That's the entire training objective.

In [ ]:
dataloader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=2, drop_last=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.mixed_precision)

def save_checkpoint(epoch):
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "ema": ema.shadow,
        "optimizer": optimizer.state_dict(),
    }, os.path.join(CKPT_DIR, f"ckpt_epoch{epoch}.pt"))
    print(f"Saved checkpoint at epoch {epoch}")

def load_latest_checkpoint():
    ckpts = [f for f in os.listdir(CKPT_DIR) if f.startswith("ckpt_epoch")]
    if not ckpts:
        return 0
    latest = max(ckpts, key=lambda f: int(f.split("epoch")[1].split(".")[0]))
    state = torch.load(os.path.join(CKPT_DIR, latest), map_location=device)
    model.load_state_dict(state["model"])
    ema.shadow = state["ema"]
    optimizer.load_state_dict(state["optimizer"])
    print(f"Resumed from {latest}")
    return state["epoch"] + 1

start_epoch = load_latest_checkpoint()  # resumes automatically if a checkpoint exists


In [ ]:
for epoch in range(start_epoch, cfg.epochs):
    model.train()
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
    running_loss = 0.0

    for step, (x0, _) in enumerate(pbar):
        x0 = x0.to(device)
        b = x0.shape[0]
        t = torch.randint(0, cfg.timesteps, (b,), device=device).long()
        noise = torch.randn_like(x0)
        x_t = q_sample(x0, t, noise)

        with torch.cuda.amp.autocast(enabled=cfg.mixed_precision):
            predicted_noise = model(x_t, t)
            loss = F.mse_loss(predicted_noise, noise)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        running_loss += loss.item()
        pbar.set_postfix(loss=running_loss / (step + 1))

    save_checkpoint(epoch)


## 6. Sampling (reverse process)

Start from pure Gaussian noise and iteratively denoise, using the model's noise prediction
at each step, for `T` steps. This is the slow, "full DDPM" sampler (1000 steps).

In [ ]:
@torch.no_grad()
def p_sample(model, x_t, t_index):
    t = torch.full((x_t.shape[0],), t_index, device=device, dtype=torch.long)
    beta_t = extract(betas, t, x_t.shape)
    sqrt_omac_t = extract(sqrt_one_minus_alphas_cumprod, t, x_t.shape)
    sqrt_recip_alpha_t = extract(torch.sqrt(1.0 / alphas), t, x_t.shape)

    predicted_noise = model(x_t, t)
    model_mean = sqrt_recip_alpha_t * (x_t - beta_t * predicted_noise / sqrt_omac_t)

    if t_index == 0:
        return model_mean
    else:
        posterior_var_t = extract(posterior_variance, t, x_t.shape)
        noise = torch.randn_like(x_t)
        return model_mean + torch.sqrt(posterior_var_t) * noise

@torch.no_grad()
def sample(model, n_samples=16):
    model.eval()
    x = torch.randn(n_samples, cfg.channels, cfg.image_size, cfg.image_size, device=device)
    for t_index in tqdm(reversed(range(cfg.timesteps)), total=cfg.timesteps, desc="Sampling"):
        x = p_sample(model, x, t_index)
    return x

# Sample using EMA weights for best quality
eval_model = UNet(cfg).to(device)
eval_model.load_state_dict(ema.shadow)
samples = sample(eval_model, n_samples=16)

grid = (samples.clamp(-1, 1) + 1) / 2
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(grid[i].permute(1, 2, 0).cpu())
    ax.axis("off")
plt.suptitle("Generated CIFAR-10-like samples")
plt.show()


## 7. (Stretch) DDIM fast sampling

Full DDPM sampling takes 1000 network evaluations per image. DDIM reformulates the same trained
model to sample deterministically in far fewer steps (e.g. 50), by skipping most timesteps —
a good illustration that the *training* objective and the *sampling* procedure are somewhat decoupled.

In [ ]:
@torch.no_grad()
def ddim_sample(model, n_samples=16, ddim_steps=50, eta=0.0):
    model.eval()
    step_indices = torch.linspace(0, cfg.timesteps - 1, ddim_steps, dtype=torch.long).flip(0)
    x = torch.randn(n_samples, cfg.channels, cfg.image_size, cfg.image_size, device=device)

    for i in tqdm(range(len(step_indices)), desc="DDIM sampling"):
        t_val = step_indices[i].item()
        t = torch.full((n_samples,), t_val, device=device, dtype=torch.long)
        predicted_noise = model(x, t)

        alpha_bar_t = alphas_cumprod[t_val]
        alpha_bar_prev = alphas_cumprod[step_indices[i + 1].item()] if i + 1 < len(step_indices) else torch.tensor(1.0, device=device)

        x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * predicted_noise) / torch.sqrt(alpha_bar_t)
        x0_pred = x0_pred.clamp(-1, 1)

        dir_xt = torch.sqrt(1 - alpha_bar_prev) * predicted_noise
        x = torch.sqrt(alpha_bar_prev) * x0_pred + dir_xt

    return x

ddim_samples = ddim_sample(eval_model, n_samples=16, ddim_steps=50)
grid = (ddim_samples.clamp(-1, 1) + 1) / 2
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(grid[i].permute(1, 2, 0).cpu())
    ax.axis("off")
plt.suptitle("DDIM samples (50 steps vs 1000)")
plt.show()


## Next steps
- **Class-conditioning**: add CIFAR-10's 10 class labels as conditioning (embed class -> add to time embedding). This is the stepping stone to text-conditioning.
- **Text-conditioning**: swap class embedding for a text encoder (e.g. CLIP text encoder, frozen) + cross-attention in the U-Net, on a narrow-domain captioned dataset (e.g. Flowers-102).
- **Better schedules**: try a cosine beta schedule instead of linear — known to improve sample quality.
